# Stage 8a — Download the Full Predictor Stack

Pull every additional predictor for the same box & 3-year window we've used since Notebook 1.

## What we download

| Group | Source | Variables | Why |
|---|---|---|---|
| Wind | **ERA5** (CDS) | `u10`, `v10` | Atmospheric forcing — coastal upwelling driver. Basin-scale signal. |
| Salinity | CMEMS PHY MY | `so` | Surface salinity — Atlantic-Ionian Stream (AIS) marker. |
| Currents | CMEMS PHY MY | `uo`, `vo` | Advection — dominant offshore driver per literature. |
| Sea-surface height | CMEMS PHY MY | `zos` | AIS proxy via dynamic topography. |
| Mixed-layer depth | CMEMS PHY MY | `mlotst` | Stratification → light availability → bloom potential. |
| KD490 | CMEMS OC MY L3 | `KD490` | Light attenuation — direct ocean-colour optical depth. |

## What we deliberately skip (this loop)

- **Waves** (`MEDSEA_MULTIYEAR_WAV_006_012`) — only hourly product available; ~1.7 GB download to aggregate locally. Defer until 8b ablation shows whether we need them.
- **PAR** — no MY L3 PAR product exists in the Med catalogue; would need to be derived from ERA5 solar radiation. Defer.
- **BGC nutrients** (`MEDSEA_MULTIYEAR_BGC_006_008`) — optional per PLAN.md. Defer.
- **ASCAT wind enrichment** — ERA5 is the workhorse; ASCAT adds satellite-direct refinement. Defer.

## Why these choices

PLAN.md's strategy: get the pipeline working with a sensible stack, then let **group ablation** in Notebook 8b tell us which groups are pulling their weight. Adding everything upfront is the opposite of disciplined feature engineering — it lets noisy variables hide the signal of useful ones, and makes the writeup unfocused.

## 8a.0 — Configuration

In [1]:
from pathlib import Path

BOX = dict(
    minimum_longitude=12.0, maximum_longitude=15.5,
    minimum_latitude=35.5,  maximum_latitude=37.5,
)

START = "2021-01-01T00:00:00"
END   = "2023-12-31T00:00:00"

RAW_DIR = Path("data/raw"); RAW_DIR.mkdir(parents=True, exist_ok=True)

FORCE = False   # set True to re-download regardless of cache

# CMEMS jobs: (dataset_id, variables, output filename, label)
CMEMS_JOBS = [
    ("cmems_mod_med_phy-sal_my_4.2km_P1D-m",         ["so"],         "phy_sal.nc",     "salinity"),
    ("cmems_mod_med_phy-cur_my_4.2km_P1D-m",         ["uo", "vo"],   "phy_cur.nc",     "currents"),
    ("cmems_mod_med_phy-ssh_my_4.2km_P1D-m",         ["zos"],        "phy_ssh.nc",     "ssh"),
    ("cmems_mod_med_phy-mld_my_4.2km_P1D-m",         ["mlotst"],     "phy_mld.nc",     "mld"),
    ("cmems_obs-oc_med_bgc-transp_my_l3-multi-1km_P1D", ["KD490"],  "oc_kd490.nc",    "kd490"),
]

ERA5_PATH = RAW_DIR / "era5_wind_2021_2023.nc"

print("Box     :", BOX)
print("Period  :", START, "→", END)
print("CMEMS jobs:")
for ds_id, vars_, fn, label in CMEMS_JOBS:
    print(f"   {label:10s} {ds_id}  →  {fn}")
print(f"ERA5    : {ERA5_PATH.name}")

Box     : {'minimum_longitude': 12.0, 'maximum_longitude': 15.5, 'minimum_latitude': 35.5, 'maximum_latitude': 37.5}
Period  : 2021-01-01T00:00:00 → 2023-12-31T00:00:00
CMEMS jobs:
   salinity   cmems_mod_med_phy-sal_my_4.2km_P1D-m  →  phy_sal.nc
   currents   cmems_mod_med_phy-cur_my_4.2km_P1D-m  →  phy_cur.nc
   ssh        cmems_mod_med_phy-ssh_my_4.2km_P1D-m  →  phy_ssh.nc
   mld        cmems_mod_med_phy-mld_my_4.2km_P1D-m  →  phy_mld.nc
   kd490      cmems_obs-oc_med_bgc-transp_my_l3-multi-1km_P1D  →  oc_kd490.nc
ERA5    : era5_wind_2021_2023.nc


## 8a.1 — Verify all CMEMS dataset IDs resolve

Fail fast before any long download. If anything errors, run `copernicusmarine describe --contains "<FAMILY>"` (e.g. `MEDSEA_MULTIYEAR_PHY`) and update the `CMEMS_JOBS` table above with the current short-name.

In [2]:
import copernicusmarine

all_ok = True
for ds_id, vars_, fn, label in CMEMS_JOBS:
    try:
        copernicusmarine.describe(dataset_id=ds_id)
        print(f"OK    {label:10s} {ds_id}")
    except Exception as e:
        print(f"FAIL  {label:10s} {ds_id}\n      {e}")
        all_ok = False

if not all_ok:
    raise RuntimeError("One or more CMEMS dataset_ids did not resolve — see above and fix CMEMS_JOBS before re-running.")

Fetching catalogue 1: 100%|██████████| 2/2 [00:00<00:00,  2.28it/s]


OK    salinity   cmems_mod_med_phy-sal_my_4.2km_P1D-m


Fetching catalogue 1: 100%|██████████| 2/2 [00:00<00:00,  2.45it/s]


OK    currents   cmems_mod_med_phy-cur_my_4.2km_P1D-m


Fetching catalogue 1: 100%|██████████| 2/2 [00:00<00:00,  2.44it/s]


OK    ssh        cmems_mod_med_phy-ssh_my_4.2km_P1D-m


Fetching catalogue 1: 100%|██████████| 2/2 [00:00<00:00,  2.38it/s]


OK    mld        cmems_mod_med_phy-mld_my_4.2km_P1D-m


Fetching catalogue 1:  50%|█████     | 1/2 [00:00<00:00,  1.33it/s]

OK    kd490      cmems_obs-oc_med_bgc-transp_my_l3-multi-1km_P1D


## 8a.2 — Download CMEMS PHY + KD490

Loop over the job table, idempotent (skip if file exists). Each download is a few hundred MB at most, except KD490 which is on the 1 km grid (will be bigger).

If a job fails mid-download you can rerun the cell — completed files will be skipped.

In [3]:
for ds_id, vars_, fn, label in CMEMS_JOBS:
    out_path = RAW_DIR / fn
    if out_path.exists() and not FORCE:
        size_mb = out_path.stat().st_size / 1e6
        print(f"SKIP  {label:10s} {fn} ({size_mb:.1f} MB)")
        continue
    print(f"\n>>> {label}: downloading {ds_id}")
    try:
        copernicusmarine.subset(
            dataset_id=ds_id,
            variables=vars_,
            start_datetime=START,
            end_datetime=END,
            output_filename=fn,
            output_directory=str(RAW_DIR),
            **BOX,
        )
        size_mb = out_path.stat().st_size / 1e6
        print(f"OK    {label:10s} → {fn} ({size_mb:.1f} MB)")
    except Exception as e:
        print(f"FAIL  {label:10s} {ds_id}\n      {e}")


>>> salinity: downloading cmems_mod_med_phy-sal_my_4.2km_P1D-m


INFO - 2026-06-18T04:58:38Z - Selected dataset version: "202511"
INFO - 2026-06-18T04:58:38Z - Selected dataset part: "default"
Fetching catalogue 1: 100%|██████████| 2/2 [00:07<00:00,  3.92s/it]


  0%|          | [00:00<?]

INFO - 2026-06-18T05:00:06Z - Total size of the download: 2.35 GB.


OK    salinity   → phy_sal.nc (2519.8 MB)

>>> currents: downloading cmems_mod_med_phy-cur_my_4.2km_P1D-m


...

OK    mld        → phy_mld.nc (17.9 MB)

>>> kd490: downloading cmems_obs-oc_med_bgc-transp_my_l3-multi-1km_P1D


INFO - 2026-06-18T05:04:24Z - Selected dataset version: "202311"
INFO - 2026-06-18T05:04:24Z - Selected dataset part: "default"


  0%|          | [00:00<?]

INFO - 2026-06-18T05:04:47Z - Total size of the download: 225.10 MB.


OK    kd490      → oc_kd490.nc (235.9 MB)


## 8a.3 — ERA5 wind via CDS

**One-time setup required.** Skip this cell if you don't have credentials yet:

1. Register a free account at <https://cds.climate.copernicus.eu>.
2. Accept the **ERA5 reanalysis** licence on the dataset page (click 'Show terms' → 'Accept'). Without this, requests will fail with `Required license not accepted`.
3. Find your API key on your profile page.
4. Create `~/.cdsapirc` (on Windows: `C:\Users\<you>\.cdsapirc`) with two lines:
   ```
   url: https://cds.climate.copernicus.eu/api
   key: <YOUR-API-KEY>
   ```

**What we pull:** `10m_u_component_of_wind`, `10m_v_component_of_wind` at 12:00 UTC daily (one snapshot/day) for our box. ~50–100 MB. ERA5 queue can take 5–30 minutes at peak times — the call blocks until the file is ready.

Why noon? It's the standard midday reference for daily wind in air-sea-flux work, and avoids diurnal bias (sea-breeze peaks ~14:00, land cools at night).

In [5]:
SKIP_ERA5 = False   # flip to True if you can't / don't want to set up CDS now

if SKIP_ERA5:
    print("SKIP_ERA5=True — ERA5 download deferred. Wind features will be missing in 8b.")
elif ERA5_PATH.exists() and not FORCE:
    print(f"SKIP era5 {ERA5_PATH.name} ({ERA5_PATH.stat().st_size/1e6:.1f} MB)")
else:
    import cdsapi
    c = cdsapi.Client()
    print("Submitting ERA5 request — this may queue for 5-30 minutes…")
    c.retrieve(
        "reanalysis-era5-single-levels",
        {
            "product_type": "reanalysis",
            "variable": ["10m_u_component_of_wind", "10m_v_component_of_wind"],
            "year":  ["2021", "2022", "2023"],
            "month": [f"{m:02d}" for m in range(1, 13)],
            "day":   [f"{d:02d}" for d in range(1, 32)],
            "time":  ["12:00"],
            "area":  [BOX["maximum_latitude"], BOX["minimum_longitude"],
                       BOX["minimum_latitude"], BOX["maximum_longitude"]],  # N, W, S, E
            "format": "netcdf",
        },
        str(ERA5_PATH),
    )
    print(f"Saved → {ERA5_PATH}  ({ERA5_PATH.stat().st_size/1e6:.1f} MB)")

Submitting ERA5 request — this may queue for 5-30 minutes…


2026-06-18 05:07:10,078 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=overview)
2026-06-18 05:07:10,079 INFO Request ID is d81279d6-1dbd-4d83-abec-da0cf8df2472
2026-06-18 05:07:10,153 INFO status has been updated to accepted
2026-06-18 05:07:39,208 INFO status has been updated to running
2026-06-18 05:09:12,388 INFO status has been updated to successful


a99254bbe687476aa58f82ac82feb31e.nc:   0%|          | 0.00/830k [00:00<?, ?B/s]

Saved → data\raw\era5_wind_2021_2023.nc  (0.8 MB)


## 8a.4 — Inspect every download

Quick sanity-check loop: dims, time range, value range, units. Catches half the silent bugs before they propagate into 8b.

In [6]:
import xarray as xr
import numpy as np

def quick_inspect(path, label):
    if not path.exists():
        print(f"--- {label}: MISSING ({path}) ---")
        return
    ds = xr.open_dataset(path)
    print(f"\n--- {label} ({path.name}, {path.stat().st_size/1e6:.1f} MB) ---")
    print(f"  dims  : {dict(ds.sizes)}")
    if "time" in ds.dims:
        tmin, tmax = ds.time.values.min(), ds.time.values.max()
        print(f"  time  : {str(tmin)[:10]} → {str(tmax)[:10]}")
    for v in ds.data_vars:
        a = ds[v]
        if a.dtype.kind in "fui":
            sample = a.isel({d: 0 for d in a.dims if d != "time"}).values
            if a.ndim >= 2:
                sample = a.isel(time=0).values if "time" in a.dims else a.values
            try:
                mn, md, mx = float(np.nanmin(sample)), float(np.nanmedian(sample)), float(np.nanmax(sample))
                print(f"  {v:8s} {a.attrs.get('units','')[:12]:12s} min={mn:.3f}  med={md:.3f}  max={mx:.3f}")
            except Exception:
                pass

for ds_id, vars_, fn, label in CMEMS_JOBS:
    quick_inspect(RAW_DIR / fn, label)

quick_inspect(ERA5_PATH, "era5_wind")


--- salinity (phy_sal.nc, 2519.8 MB) ---
  dims  : {'time': 1095, 'depth': 141, 'latitude': 48, 'longitude': 85}
  time  : 2021-01-01 → 2023-12-31
  so       0.001        min=37.301  med=38.510  max=38.975

--- currents (phy_cur.nc, 5039.5 MB) ---
  dims  : {'time': 1095, 'depth': 141, 'latitude': 48, 'longitude': 85}
  time  : 2021-01-01 → 2023-12-31
  uo       m s-1        min=-0.302  med=0.002  max=0.521
  vo       m s-1        min=-0.443  med=0.001  max=0.276

--- ssh (phy_ssh.nc, 17.9 MB) ---
  dims  : {'time': 1095, 'latitude': 48, 'longitude': 85}
  time  : 2021-01-01 → 2023-12-31
  zos      m            min=-0.483  med=-0.336  max=-0.269

--- mld (phy_mld.nc, 17.9 MB) ---
  dims  : {'time': 1095, 'latitude': 48, 'longitude': 85}
  time  : 2021-01-01 → 2023-12-31
  mlotst   m            min=6.264  med=64.060  max=123.466

--- kd490 (oc_kd490.nc, 235.9 MB) ---
  dims  : {'time': 1095, 'latitude': 198, 'longitude': 272}
  time  : 2021-01-01 → 2023-12-31
  KD490    m^-1         mi

---
## Validation Gate 8a

Three things must hold before moving to 8b:

1. **All five CMEMS files exist** (ERA5 may be deferred but the CMEMS stack is required for the methodological story).
2. **Each file's time range overlaps** the CHL period by ≥ 1 year.
3. **Each file's bbox covers** the CHL box (lon 12–15.5, lat 35.5–37.5).

In [7]:
CHL_PATH = RAW_DIR / "chl_malta_2021_2023.nc"
chl = xr.open_dataset(CHL_PATH).CHL
chl_t0, chl_t1 = chl.time.values.min(), chl.time.values.max()

def lat_lon_of(ds):
    lat_name = "latitude" if "latitude" in ds.coords else "lat"
    lon_name = "longitude" if "longitude" in ds.coords else "lon"
    return ds[lat_name], ds[lon_name]

results = {}

for ds_id, vars_, fn, label in CMEMS_JOBS:
    p = RAW_DIR / fn
    results[f"{label}: file present"] = p.exists() and p.stat().st_size > 100_000
    if not p.exists():
        continue
    ds = xr.open_dataset(p)
    t0, t1 = ds.time.values.min(), ds.time.values.max()
    overlap_start = max(chl_t0, t0); overlap_end = min(chl_t1, t1)
    overlap_days = (overlap_end - overlap_start) / np.timedelta64(1, "D")
    results[f"{label}: overlap > 365d"] = overlap_days > 365

    lat, lon = lat_lon_of(ds)
    results[f"{label}: bbox covers box"] = (
        float(lon.min()) <= 12.1 and float(lon.max()) >= 15.4 and
        float(lat.min()) <= 35.6 and float(lat.max()) >= 37.4
    )
    expected_vars = set(vars_)
    actual_vars = set(ds.data_vars)
    results[f"{label}: variable present"] = expected_vars.issubset(actual_vars)

# ERA5 — soft gate, fine if missing as long as flagged
era5_ok = ERA5_PATH.exists()
results["era5: present (soft)"] = era5_ok
if era5_ok:
    ds = xr.open_dataset(ERA5_PATH)
    lat, lon = lat_lon_of(ds)
    results["era5: bbox covers box"] = (
        float(lon.min()) <= 12.1 and float(lon.max()) >= 15.4 and
        float(lat.min()) <= 35.6 and float(lat.max()) >= 37.4
    )

for name, ok in results.items():
    marker = "PASS" if ok else "FAIL"
    if name.endswith("(soft)") and not ok:
        marker = "SOFT-FAIL"
    print(f"{marker:9s} {name}")

hard_results = {k: v for k, v in results.items() if not k.endswith("(soft)")}
if all(hard_results.values()):
    print("\nGate 8a PASSED — proceed to 08b-Group-Importance-Ablation.")
    if not era5_ok:
        print("  (ERA5 deferred — wind features will be absent in 8b; you can rerun this notebook later to add them.)")
else:
    print("\nGate 8a FAILED — fix the FAILs above (usually a dataset_id drift) before continuing.")

PASS      salinity: file present
PASS      salinity: overlap > 365d
PASS      salinity: bbox covers box
PASS      salinity: variable present
PASS      currents: file present
PASS      currents: overlap > 365d
PASS      currents: bbox covers box
PASS      currents: variable present
PASS      ssh: file present
PASS      ssh: overlap > 365d
PASS      ssh: bbox covers box
PASS      ssh: variable present
PASS      mld: file present
PASS      mld: overlap > 365d
PASS      mld: bbox covers box
PASS      mld: variable present
PASS      kd490: file present
PASS      kd490: overlap > 365d
PASS      kd490: bbox covers box
PASS      kd490: variable present
PASS      era5: present (soft)
PASS      era5: bbox covers box

Gate 8a PASSED — proceed to 08b-Group-Importance-Ablation.


---
## Next: `08b-Group-Importance-Ablation.ipynb`

What 8b does, in order:
1. Regrid every predictor onto the 1 km CHL grid (the finest — we never downgrade the target).
2. Engineer derived wind features (wind speed, wind-stress curl, Bakun-style upwelling index).
3. Build the full feature matrix and **correlation heatmap** (`|ρ| > 0.9` pairs get flagged for merging).
4. Retrain XGBoost per horizon on the expanded stack.
5. **Grouped permutation importance** — read importance by theme ("all wind features together"), not by individual column. This is the trap most ablation studies fall into.
6. **Group ablation** — drop each group, retrain, measure the val-R² drop. The group whose removal barely changes R² isn't earning its place.
7. Keep only the groups that ablation validates. Document the final feature set.

Outcome: a feature set you can defend in the writeup, where every variable has both individual-importance *and* ablation evidence for inclusion.